# `ofr(series, start, end=None)` and `ofr_series()`

US repo rates and volume, and money market fund assets, from `indicators.ofr`: the Office of Financial Research's Short-term Funding Monitor, oldest first. `ofr_series()` is the map from each series id to its dataset, name, units, frequency and category. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

A series is named by its OFR mnemonic (`REPO-TRI_AR_OO-P`) or by its name, in any case and spacing. Days with no value are left out, never filled.

> **About the data shown.** Source: Office of Financial Research, "OFR Short-term Funding Monitor," refreshed daily, https://www.financialresearch.gov/short-term-funding-monitor/ (accessed November 24, 2026). One change, by arora_data: the dollar amounts are OFR's figures divided by 1,000,000, so they are in USD millions; rates are as OFR publishes them.
>
> OFR's disclaimer: "This OFR monitor is presented solely for informative purposes and should not be relied upon for financial decisions; it is not intended to provide any investment or financial advice." Its repo rates are "not intended to be referenced in contracts." arora_data is not affiliated with or endorsed by the OFR.

In [1]:
from datetime import date, timedelta
from decimal import Decimal

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.ofr import OfrValue
from arora_data.sync import ofr, ofr_series


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The series
Every series the table holds, read from it on every call. The labels come from each series' latest row; `first_date` and `latest_date` are its first and latest days with a value.

In [2]:
show(ofr_series())

5 rows

series_id|dataset|name|units|frequency|category|first_date|latest_date
---|---|---|---|---|---|---|---
MMF-MMF_TOT-M|ofr/short_term_funding|Money Market Funds — Total Assets|USD millions|monthly|Money Markets — MMF|2010-11-30|2026-10-31
REPO-DVP_AR_OO-P|ofr/short_term_funding|DVP Repo — Avg Rate, Overnight|percent|daily|Money Markets — Repo|2018-05-08|2026-11-20
REPO-GCF_AR_AG-P|ofr/short_term_funding|GCF Repo — Avg Rate, All|percent|daily|Money Markets — Repo|2018-05-07|2026-11-20
REPO-TRI_AR_OO-P|ofr/short_term_funding|Tri-Party Repo — Avg Rate, Overnight|percent|daily|Money Markets — Repo|2014-08-22|2026-11-19
REPO-TRI_TV_TOT-P|ofr/short_term_funding|Tri-Party Repo — Total Volume|USD millions|daily|Money Markets — Repo|2014-08-22|2026-11-19

## One series, by id or by name
The tri-party repo overnight rate over the last two weeks, in percent. The id and the name, in any case and spacing, give the same rows. Each row is a business day: weekends and holidays have none.

In [3]:
two_weeks = today_et() - timedelta(days=14)
assert ofr("REPO-TRI_AR_OO-P", two_weeks) == ofr("repo-tri_ar_oo-p", two_weeks) \
    == ofr(" tri-party repo — AVG rate,  overnight ", two_weeks)
show(ofr("REPO-TRI_AR_OO-P", two_weeks))

7 rows

series_id|name|date|value|units
---|---|---|---|---
REPO-TRI_AR_OO-P|Tri-Party Repo — Avg Rate, Overnight|2026-11-10|3.88000000|percent
REPO-TRI_AR_OO-P|Tri-Party Repo — Avg Rate, Overnight|2026-11-12|3.88000000|percent
REPO-TRI_AR_OO-P|Tri-Party Repo — Avg Rate, Overnight|2026-11-13|3.88000000|percent
REPO-TRI_AR_OO-P|Tri-Party Repo — Avg Rate, Overnight|2026-11-16|3.91000000|percent
REPO-TRI_AR_OO-P|Tri-Party Repo — Avg Rate, Overnight|2026-11-17|3.91000000|percent
REPO-TRI_AR_OO-P|Tri-Party Repo — Avg Rate, Overnight|2026-11-18|3.89000000|percent
REPO-TRI_AR_OO-P|Tri-Party Repo — Avg Rate, Overnight|2026-11-19|3.89000000|percent

## The three repo rates, side by side
DVP (bilateral, centrally cleared), GCF (general collateral, centrally cleared) and tri-party, in percent, joined on date. A day one segment has no value shows `null`: tri-party comes out a business day after DVP and GCF, so its latest day is one behind.

In [4]:
rates = {"REPO-DVP_AR_OO-P": "DVP", "REPO-GCF_AR_AG-P": "GCF", "REPO-TRI_AR_OO-P": "tri-party"}
frames = [to_df(ofr(i, two_weeks), OfrValue).select("date", pl.col("value").alias(n)) for i, n in rates.items()]
joined = frames[0]
for f in frames[1:]:
    joined = joined.join(f, on="date", how="full", coalesce=True)
joined.sort("date")

date,DVP,GCF,tri-party
date,"decimal[38,8]","decimal[38,8]","decimal[38,8]"
2026-11-10,3.91000000,3.94000000,3.88000000
2026-11-12,3.92000000,3.95000000,3.88000000
2026-11-13,3.91000000,3.94000000,3.88000000
2026-11-16,3.94000000,3.96000000,3.91000000
2026-11-17,3.94000000,3.97000000,3.91000000
2026-11-18,3.92000000,3.95000000,3.89000000
2026-11-19,3.92000000,3.96000000,3.89000000
2026-11-20,3.93000000,3.94000000,null


## Units
Rates are in percent: 3.91 is 3.91%. Tri-party volume and money market fund assets are in USD millions: 3951744.1 is about $3.95 trillion. Their latest values, in USD trillions:

In [5]:
for i in ["REPO-TRI_TV_TOT-P", "MMF-MMF_TOT-M"]:
    v = ofr(i, today_et() - timedelta(days=62))[-1]
    print(f"{v.name:<36} {v.date}  {(v.value / Decimal(10**6)).quantize(Decimal('0.001')):>7}")

Tri-Party Repo — Total Volume        2026-11-19    2.280


Money Market Funds — Total Assets    2026-10-31    8.473


## Money market fund assets, monthly
One value per month, dated at the month's last calendar day, from 2010-11-30. Assets at each year's last month, in USD trillions:

In [6]:
mmf = to_df(ofr("MMF-MMF_TOT-M", date(1900, 1, 1)), OfrValue)
print(f"{mmf.height} months, {mmf['date'].min()} to {mmf['date'].max()}")
(mmf.group_by(pl.col("date").dt.year().alias("year")).agg(pl.col("value").sort_by("date").last())
    .with_columns((pl.col("value") / 1_000_000).round(2).alias("USD trillions")).drop("value").sort("year"))

192 months, 2010-11-30 to 2026-10-31


year,USD trillions
i32,"decimal[38,8]"
2010,3.04000000
2011,2.91000000
2012,2.98000000
2013,3.05000000
2014,3.08000000
…,…
2022,5.21000000
2023,6.40000000
2024,7.25000000


## Days OFR left blank
OFR publishes no value for a day that fails its disclosure threshold, which protects business-confidential data. The table stores those days as NULL, and `ofr()` leaves them out. DVP has 96 such days, from 2018-05-07 to 2025-02-06. Weekdays without a DVP value per year, where the GCF rate has one:

In [7]:
gcf = to_df(ofr("REPO-GCF_AR_AG-P", date(1900, 1, 1)), OfrValue).select("date")
dvp = to_df(ofr("REPO-DVP_AR_OO-P", date(1900, 1, 1)), OfrValue)
gcf.join(dvp, on="date", how="anti").group_by(pl.col("date").dt.year().alias("year")).len().sort("year")

year,len
i32,u32
2018,8
2019,6
2020,16
2021,13
2022,16
2023,12
2024,22
2025,3


Tri-party's rate also has no row on 2022-06-17, a day its volume has:

In [8]:
day = date(2022, 6, 17)
print("rate:", ofr("REPO-TRI_AR_OO-P", day, day), " volume:", [v.value for v in ofr("REPO-TRI_TV_TOT-P", day, day)])

rate: []  volume: [Decimal('3414528.21919511')]


## Rates below zero
DVP went below zero on 94 days, from 2020-03-24 to 2022-03-14, as OFR publishes it. The values are served as stored. Days below zero, per year:

In [9]:
dvp.filter(pl.col("value") < 0).group_by(pl.col("date").dt.year().alias("year")) \
    .agg(pl.len().alias("days"), pl.col("value").min().alias("lowest")).sort("year")

year,days,lowest
i32,u32,"decimal[38,8]"
2020,9,-0.02000000
2021,83,-0.08000000
2022,2,-0.02000000


## Revisions
The main repo re-reads each series' whole history on every run, so OFR's revisions to its preliminary values replace the earlier ones, and earlier values are not stored.

## Not here
SOFR, TGCR and BGCR, the New York Fed's repo reference rates, are `nyfed()`.

## Bad input
An unknown id or name, or a start after the end, raises a clear error before any values are read.

In [10]:
for args in [("sofr", date(2026, 1, 1)), ("DVP Repo", date(2026, 1, 1)), (None, date(2026, 1, 1)),
             ("REPO-DVP_AR_OO-P", date(2026, 1, 2), date(2026, 1, 1))]:
    try:
        ofr(*args)
    except ValueError as e:
        print(e)

'sofr' is not a stored OFR series: use an id or name from ofr_series(), such as REPO-TRI_AR_OO-P or 'GCF Repo — Avg Rate, All'


'DVP Repo' is not a stored OFR series: use an id or name from ofr_series(), such as REPO-TRI_AR_OO-P or 'GCF Repo — Avg Rate, All'


None is not a stored OFR series: use an id or name from ofr_series(), such as REPO-TRI_AR_OO-P or 'GCF Repo — Avg Rate, All'
start must be on or before end
